# Assignment 1A: build and fine-tune a domain LLM

**Continual Pre-Training (CPT) → QLoRA instruction fine-tuning · 15 marks**

| | |
|---|---|
| **Domain** | Variant 5: IT Helpdesk & Incident Resolution (RHEL 9 admin guides + Ubuntu Server guide) |
| **Model** | `Qwen/Qwen2.5-0.5B` (base model, 494M parameters; fits a free Colab T4) |
| **Run on** | Any NVIDIA GPU: Colab (*Runtime → Change runtime type → T4 GPU*), the institute GPU server, or a rented GPU |

Open **`explainer.html`** first. Each section below says which explainer part it uses.

**How this notebook works:** the boilerplate is already written. The lines that carry the main idea are left as `# TODO n`.
Write them, run the cell, and the `assert` checks tell you whether you got them right.
Under every step there is a **✍️ Your inference** cell. The assignment says these are *mandatory*, so do not skip them.

| Part | Step | Marks |
|---|---|---|
| A | 1 Data collection, extraction, cleaning | 2 |
| A | 2 Tokenization and packed dataset | 2 |
| A | 3 Model loading and architecture | 2 |
| A | 4 CPT training and loss curve | 2 |
| A | 5 Perplexity and catastrophic forgetting | 2 |
| B | B1 Instruction dataset | 2 |
| B | B2 QLoRA fine-tuning (one adapter) | 2 |
| B | B3 Evaluation | 1 |

---
## Cell 0: Setup

Installs the libraries and sets the shared settings. Nothing to write here.

On Colab, everything is saved to your Google Drive, because Colab wipes its disk when the session ends and Part B needs
the CPT checkpoint from Step 4. On the institute GPU server (or any other machine) it saves under your home folder.
The cell detects which one you are on.

In [ ]:
%pip install -q -U transformers peft trl bitsandbytes accelerate datasets pymupdf langdetect

In [ ]:
import os, re, gc, json, math, time, random, hashlib, collections
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Colab -> save to Google Drive (survives disconnects). Anywhere else (institute GPU server,
# rented machine) -> save under your home folder.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = Path('/content/drive/MyDrive/llm_assignment_1a')
else:
    WORKDIR = Path.home() / 'llm_assignment_1a'

PDF_DIR     = WORKDIR / 'domain_pdfs'      # downloaded PDFs
RAW_DIR     = WORKDIR / 'raw_txt'          # text straight out of the PDFs
CORPUS_DIR  = WORKDIR / 'domain_corpus'    # cleaned text  (deliverable 4)
DATA_DIR    = WORKDIR / 'data'             # parquet + jsonl
CPT_DIR     = WORKDIR / 'cpt_model'        # checkpoint after Step 4
ADAPTER_DIR = WORKDIR / 'qlora_adapter_B'  # adapter after B2
for d in [PDF_DIR, RAW_DIR, CORPUS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_ID   = 'Qwen/Qwen2.5-0.5B'
BLOCK_SIZE = 512          # tokens per packed training sequence
SEED       = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
# bfloat16 needs an Ampere-or-newer GPU (A100, L4). A T4 is older, so on a T4 we fall back (see Step 3).
BF16_OK = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (switch the runtime to GPU!)')
print('bf16 supported:', BF16_OK)

# The same 3 domain prompts are used in Step 3 (baseline), Step 5 (CPT) and B3 (adapter).
DOMAIN_PROMPTS = [
    'How do I permanently open TCP port 8080 in firewalld on RHEL 9?',
    'How do I configure a static IPv4 address on a network interface using nmcli?',
    'How can I find out which package provides a specific file using dnf?',
]
GENERAL_PROMPTS = [
    'The capital of France is',
    'Water boils at',
    'The speed of light is approximately',
]

---
# PART A: Continual Pre-Training (CPT)

## Step 1: Data collection, extraction and cleaning [2 marks] · *explainer §2*

### 1a. Collect the PDFs
These are 24 free Red Hat Enterprise Linux 9 guides plus the Ubuntu Server guide, all public documentation.
Already-downloaded files are skipped, so you can re-run the cell safely.

In [ ]:
import urllib.request

RHEL = 'https://docs.redhat.com/en/documentation/red_hat_enterprise_linux/9/pdf/{s}/Red_Hat_Enterprise_Linux-9-{S}-en-US.pdf'
RHEL_GUIDES = [
    'configuring_basic_system_settings', 'configuring_firewalls_and_packet_filters',
    'configuring_and_managing_networking', 'managing_storage_devices', 'managing_file_systems',
    'configuring_and_managing_logical_volumes', 'managing_software_with_the_dnf_tool',
    'securing_networks', 'security_hardening', 'using_selinux',
    'configuring_authentication_and_authorization_in_rhel', 'managing_monitoring_and_updating_the_kernel',
    'monitoring_and_managing_system_status_and_performance', 'deploying_web_servers_and_reverse_proxies',
    'managing_networking_infrastructure_services', 'configuring_and_using_network_file_services',
    'automating_system_administration_by_using_rhel_system_roles', 'managing_systems_using_the_rhel_9_web_console',
    'building_running_and_managing_containers', 'configuring_and_managing_virtualization',
    'managing_and_monitoring_security_updates', 'upgrading_from_rhel_8_to_rhel_9',
    'deploying_mail_servers', 'configuring_and_using_database_servers',
]
SOURCES = {f'rhel9_{s}': RHEL.format(s=s, S=s[0].upper() + s[1:]) for s in RHEL_GUIDES}
SOURCES['ubuntu_server_guide'] = 'https://documentation.ubuntu.com/server/_/downloads/en/latest/pdf/'

for name, url in SOURCES.items():
    out = PDF_DIR / f'{name}.pdf'
    if out.exists() and out.stat().st_size > 10_000:
        continue
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        out.write_bytes(urllib.request.urlopen(req, timeout=120).read())
        print('downloaded', name)
    except Exception as e:
        print('FAILED', name, e)

pdfs = sorted(PDF_DIR.glob('*.pdf'))
print(f'\n{len(pdfs)} PDFs, {sum(p.stat().st_size for p in pdfs)/1e6:.1f} MB')

### 1b. Extract text page by page
`PyMuPDF` (imported as `fitz`) is a standard PDF text extractor. We save one `.txt` per PDF in `raw_txt/`,
with pages separated by a form-feed character `\f` so we can split them apart again later.

In [ ]:
import fitz  # PyMuPDF

def extract_pages(pdf_path):
    # Return a list with one string per page.
    with fitz.open(pdf_path) as doc:
        # TODO 1: return [ ... ] the text of every page. Hint: page.get_text() for page in doc
        return None

raw_docs = {}
for p in pdfs:
    pages = extract_pages(p)
    raw_docs[p.stem] = pages
    (RAW_DIR / f'{p.stem}.txt').write_text('\f'.join(pages), encoding='utf-8')

assert all(isinstance(v, list) and len(v) > 0 for v in raw_docs.values()), 'extract_pages must return a non-empty list of strings'
print(len(raw_docs), 'documents,', sum(len(v) for v in raw_docs.values()), 'pages extracted')

### 1c. Clean the text: normalise, then four filters

**Normalise (written for you):** every page of a manual repeats the same header and footer lines, for example
*"Red Hat Enterprise Linux 9 Configuring firewalls..."*, page numbers and *"CHAPTER 3. ..."*. A line that appears on
at least 30% of a document's pages is boilerplate, so it is removed. Lines that a PDF wrapped mid-sentence are joined back.

Then we filter **pages**. A page is the unit here because one PDF is one big document; filtering whole PDFs would throw
away 100 good pages because of 1 bad one. A document survives as long as at least one of its pages survives.

| Filter | Removes | Your job |
|---|---|---|
| 1 Length | near-empty pages (title page, "this page left blank") | TODO 2 |
| 2 Boilerplate / TOC | table of contents, legal notice, "providing feedback" pages | written for you |
| 3 Deduplication | exact duplicate pages (after lower-casing and stripping punctuation) | TODO 3 |
| 4 Language | non-English pages | TODO 4 |

In [ ]:
BOILER_LINE = [re.compile(p) for p in [r'^\d{1,4}$', r'^CHAPTER \d+\. ', r'^APPENDIX [A-Z]\. ']]

def normalise_doc(pages):
    lines_per_page = [[l.strip() for l in p.splitlines() if l.strip()] for p in pages]
    counts = collections.Counter(l for ls in lines_per_page for l in set(ls))
    threshold = max(3, 0.3 * len(pages))
    cleaned = []
    for ls in lines_per_page:
        keep = [l for l in ls if counts[l] < threshold and not any(b.match(l) for b in BOILER_LINE)]
        merged = []
        for l in keep:   # re-join lines the PDF wrapped mid-sentence
            if merged and not re.search(r'[.:;!?]$', merged[-1]) and re.match(r'[a-z(]', l):
                merged[-1] += ' ' + l
            else:
                merged.append(l)
        cleaned.append(re.sub(r'[ \t]+', ' ', '\n'.join(merged)))
    return cleaned

# one row per page: (document name, page number, text)
pages = [(doc, i, t) for doc, ps in raw_docs.items() for i, t in enumerate(normalise_doc(ps))]

stats = []
def record(step, rows):
    stats.append({'step': step, 'documents': len({r[0] for r in rows}), 'pages': len(rows),
                  'words': sum(len(r[2].split()) for r in rows)})
record('0 raw extracted', pages)

In [ ]:
# ---- Filter 1: length --------------------------------------------------------
MIN_WORDS = 50   # justify this number in your inference (about 3-4 sentences of real content)

def long_enough(text):
    # TODO 2: return True if the page has at least MIN_WORDS words.  Hint: len(text.split())
    return None

pages = [r for r in pages if long_enough(r[2])]
record('1 length filter', pages)

# ---- Filter 2: boilerplate / table-of-contents (written for you) -------------
HEADING = re.compile(r'^\d+(?:\.\d+)+\.?\s+[A-Za-z]')
BOILER_PHRASES = ('PROVIDING FEEDBACK ON', 'Legal Notice', 'MAKING OPEN SOURCE MORE INCLUSIVE', 'Table of Contents')
def is_boilerplate(text):
    lines = text.splitlines()
    toc_ratio = sum(bool(HEADING.match(l)) for l in lines) / max(1, len(lines))
    return toc_ratio > 0.3 or any(b in text for b in BOILER_PHRASES)

pages = [r for r in pages if not is_boilerplate(r[2])]
record('2 boilerplate/TOC filter', pages)

# ---- Filter 3: exact deduplication ------------------------------------------
def fingerprint(text):
    norm = re.sub(r'\W+', ' ', text.lower()).strip()
    # TODO 3: return the md5 hex digest of `norm`.  Hint: hashlib.md5(norm.encode()).hexdigest()
    return None

seen, unique = set(), []
for r in pages:
    h = fingerprint(r[2])
    if h not in seen:
        seen.add(h)
        unique.append(r)
pages = unique
record('3 deduplication', pages)

# ---- Filter 4: language ------------------------------------------------------
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0          # langdetect is random by default; this makes it repeatable

def is_english(text):
    try:
        # TODO 4: return True if detect(...) on the first 2000 characters says 'en'
        return None
    except Exception:             # detect() raises on pages with no letters at all
        return False

pages = [r for r in pages if is_english(r[2])]
record('4 language filter (en)', pages)

stats_df = pd.DataFrame(stats)
stats_df['pages_removed'] = (-stats_df['pages'].diff()).fillna(0).astype(int)
assert stats_df['pages'].is_monotonic_decreasing and stats_df['pages'].iloc[-1] > 0, 'check TODO 2-4'
stats_df

In [ ]:
# Save the cleaned corpus: one .txt per surviving document (deliverable 4)
corpus = collections.defaultdict(list)
for doc, i, t in pages:
    corpus[doc].append(t)
for f in CORPUS_DIR.glob('*.txt'):
    f.unlink()
for doc, ps in corpus.items():
    (CORPUS_DIR / f'{doc}.txt').write_text('\n\n'.join(ps), encoding='utf-8')
print(len(corpus), 'cleaned documents saved to', CORPUS_DIR)

#### ✍️ Your inference (mandatory, marks depend on it)

- Which step removed the most pages? Why does this kind of corpus (product manuals) have that problem?
- Why did you choose MIN_WORDS = 50? What would go wrong at 5, or at 300?
- Exact hashing only catches *identical* pages. Name one kind of near-duplicate it misses and a method that would catch it (e.g. MinHash / Jaccard similarity).
- Which library did you use for language detection, and why did the language filter remove so little here?

_Write your answer here._

---
## Step 2: Tokenization and packed dataset [2 marks] · *explainer §3*

- **Reuse the model's own tokenizer.** The model's embedding table is indexed by these exact token IDs. A new tokenizer
  would give word *"firewall"* a different number, and the model would read gibberish.
- **BOS / EOS** mark where one document starts and ends inside a packed stream.
- **Packing** joins all documents into one long stream and cuts it into equal `BLOCK_SIZE` blocks, so no padding is wasted.

**Train / eval split happens here, before packing.** Step 5 needs 10% of the corpus the model never trained on, so we
hold out 10% of the *documents* now. Splitting by whole documents means no page of an eval manual leaks into training.

In [ ]:
from transformers import AutoTokenizer, AutoConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
cfg = AutoConfig.from_pretrained(MODEL_ID)

# Qwen2.5's tokenizer has no separate BOS token (tok.bos_token is None); the model config uses
# <|endoftext|> as BOS, which is also its EOS. We follow the config. Mention this in your inference.
BOS_ID = tok.bos_token_id if tok.bos_token_id is not None else cfg.bos_token_id
EOS_ID = tok.eos_token_id
print('BOS:', BOS_ID, repr(tok.convert_ids_to_tokens(BOS_ID)), '| EOS:', EOS_ID, repr(tok.eos_token))
print('vocab size (tokenizer):', len(tok))

texts = {p.stem: p.read_text(encoding='utf-8') for p in sorted(CORPUS_DIR.glob('*.txt'))}
names = sorted(texts)
random.Random(SEED).shuffle(names)
n_eval = max(1, round(0.10 * len(names)))
eval_names, train_names = sorted(names[:n_eval]), sorted(names[n_eval:])
print('eval documents:', eval_names)

In [ ]:
def doc_to_ids(text):
    # TODO 5: tokenize `text` WITHOUT the tokenizer adding its own special tokens, then wrap it:
    #         [BOS_ID] + token ids + [EOS_ID]
    #         Hint: tok(text, add_special_tokens=False)['input_ids']
    return None

def pack(id_lists, block_size):
    # TODO 6: concatenate all lists in id_lists into ONE flat list called `stream`.
    stream = None
    # TODO 7: cut `stream` into consecutive chunks of exactly block_size; drop the short leftover at the end.
    #         Hint: [stream[i:i+block_size] for i in range(0, len(stream) - block_size + 1, block_size)]
    blocks = None
    return blocks, len(stream)

train_ids = [doc_to_ids(texts[n]) for n in train_names]   # long docs print a harmless 'sequence length' warning
eval_ids  = [doc_to_ids(texts[n]) for n in eval_names]
train_blocks, train_tokens = pack(train_ids, BLOCK_SIZE)
eval_blocks,  eval_tokens  = pack(eval_ids,  BLOCK_SIZE)

assert train_ids[0][0] == BOS_ID and train_ids[0][-1] == EOS_ID, 'TODO 5: wrap with BOS and EOS'
assert all(len(b) == BLOCK_SIZE for b in train_blocks), 'TODO 7: every block must be exactly BLOCK_SIZE'
assert train_tokens // BLOCK_SIZE == len(train_blocks), 'TODO 6/7: stream length and block count disagree'

pd.DataFrame({'input_ids': train_blocks}).to_parquet(DATA_DIR / 'cpt_train.parquet')
pd.DataFrame({'input_ids': eval_blocks}).to_parquet(DATA_DIR / 'cpt_eval.parquet')

all_lens = [len(x) for x in train_ids + eval_ids]
pd.DataFrame([
    ['total tokens (train + eval)', train_tokens + eval_tokens],
    ['average document length (tokens)', round(np.mean(all_lens))],
    ['shortest / longest document (tokens)', f'{min(all_lens)} / {max(all_lens)}'],
    [f'packed train sequences ({BLOCK_SIZE} tokens each)', len(train_blocks)],
    [f'packed eval sequences ({BLOCK_SIZE} tokens each)', len(eval_blocks)],
    ['tokens dropped as leftover', (train_tokens % BLOCK_SIZE) + (eval_tokens % BLOCK_SIZE)],
], columns=['metric', 'value'])

#### ✍️ Your inference (mandatory, marks depend on it)

- Report total tokens, average document length and number of packed sequences (copy from the table).
- Why must the pretrained tokenizer be reused instead of training a new one?
- Why does packing beat padding for CPT? Roughly how much compute would padding each *page* to 512 tokens have wasted?
- Qwen2.5 uses the same token for BOS and EOS. Is that a problem? (Hint: what does the model really need to know at a boundary?)
- Why BLOCK_SIZE = 512 when Qwen2.5 supports 32K context? (Hint: T4 memory, and attention cost grows with length².)

_Write your answer here._

---
## Step 3: Model loading and architecture inspection [2 marks] · *explainer §4*

**Precision note (justify this in your inference).** The assignment suggests bfloat16 on a T4. But a T4 has no native
bf16 support, and if you *train* with the weights stored in bf16, tiny updates (learning rate 2e-5) get rounded away.
For a 0.5B model, fp32 weights fit easily (2 GB), so:
- **T4:** weights in fp32, no mixed precision. Slower, but numerically safe.
- **A100 / L4:** weights in fp32 + `bf16=True` mixed precision in Step 4 (fast *and* safe).

Gradient checkpointing (turned on in Step 4) re-computes activations during the backward pass instead of storing them,
trading about 30% more compute for much less memory.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)

# TODO 8: count trainable parameters: sum of p.numel() over model.parameters() where p.requires_grad
trainable = None
print(f'trainable parameters: {trainable:,}')

c = model.config
arch = {
    'decoder layers (num_hidden_layers)': c.num_hidden_layers,
    'attention heads (num_attention_heads)': c.num_attention_heads,
    'key/value heads (num_key_value_heads, GQA)': c.num_key_value_heads,
    'hidden size (hidden_size)': c.hidden_size,
    'head dimension (hidden_size / heads)': c.hidden_size // c.num_attention_heads,
    'FFN size (intermediate_size)': c.intermediate_size,
    'max context (max_position_embeddings)': c.max_position_embeddings,
    'vocab size (config.vocab_size)': c.vocab_size,
    'tied input/output embeddings': c.tie_word_embeddings,
}
display(pd.DataFrame(arch.items(), columns=['property', 'value']))

# TODO 9: get the output dimension of the prediction head: number of ROWS of model.lm_head.weight
lm_head_out = None
print('lm_head weight shape:', tuple(model.lm_head.weight.shape))
assert lm_head_out == c.vocab_size, 'lm_head must output one score per vocabulary entry'
print(f'lm_head outputs {lm_head_out:,} scores = config.vocab_size ✓   (tokenizer actually uses {len(tok):,})')
assert trainable > 400_000_000

In [ ]:
IM_END = tok.convert_tokens_to_ids('<|im_end|>')

@torch.no_grad()
def generate(m, prompt, max_new_tokens=150, chat=False):
    # chat=False: raw text continuation (base / CPT model). chat=True: wrap in the chat template (Part B).
    m.eval()
    text = tok.apply_chat_template([{'role': 'user', 'content': prompt}], tokenize=False,
                                   add_generation_prompt=True) if chat else prompt
    enc = tok(text, return_tensors='pt').to(m.device)
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                     eos_token_id=[tok.eos_token_id, IM_END], pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

baseline = pd.DataFrame({'prompt': DOMAIN_PROMPTS,
                         'base_model_output': [generate(model, p) for p in DOMAIN_PROMPTS]})
baseline.to_csv(DATA_DIR / 'baseline_outputs.csv', index=False)
pd.set_option('display.max_colwidth', None)
baseline

#### ✍️ Your inference (mandatory, marks depend on it)

- Report parameter count, layers, heads, hidden size and head dimension.
- lm_head outputs config.vocab_size scores, but the tokenizer has fewer tokens. Why would a model pad its vocabulary? (Hint: GPU matrix sizes that are multiples of 64/128.)
- Qwen2.5-0.5B ties its input embedding and lm_head. What does that save?
- Read the 3 baseline outputs: is the model answering, or just continuing text? Are the commands correct?

_Write your answer here._

---
## Step 4: CPT training loop and loss analysis [2 marks] · *explainer §5, §6*

The CPT objective is plain next-token prediction, the same one the model was pre-trained with (CS-2, *teacher forcing*).
So **labels are just the input IDs**. Hugging Face shifts them by one position internally, so token *t* is
scored on predicting token *t+1*.

Hyperparameters follow CS-2 slide 20 (*re-warm, then re-decay the learning rate*): a short warmup, then cosine decay,
with a small peak learning rate so the model adapts without forgetting.

In [ ]:
from transformers import Trainer, TrainingArguments, TrainerCallback, default_data_collator

class PackedDataset(torch.utils.data.Dataset):
    def __init__(self, parquet_path):
        self.blocks = pd.read_parquet(parquet_path)['input_ids'].tolist()
    def __len__(self):
        return len(self.blocks)
    def __getitem__(self, i):
        ids = torch.tensor(self.blocks[i], dtype=torch.long)
        # TODO 10: return a dict with three tensors:
        #   input_ids = ids,  attention_mask = all ones like ids,  labels = a copy of ids
        #   Hint: torch.ones_like(ids), ids.clone()
        return None

class LossRecorder(TrainerCallback):
    # Trainer calls on_log every `logging_steps`; logs is a dict like {'loss': 2.31, 'learning_rate': ...}
    def __init__(self):
        self.steps, self.losses = [], []
    def on_log(self, args, state, control, logs=None, **kwargs):
        # TODO 11: if logs contains 'loss', append state.global_step to self.steps and logs['loss'] to self.losses
        pass

train_ds = PackedDataset(DATA_DIR / 'cpt_train.parquet')
sample = train_ds[0]
assert sample is not None and set(sample) == {'input_ids', 'attention_mask', 'labels'}, 'TODO 10'
assert torch.equal(sample['labels'], sample['input_ids'])
print(len(train_ds), 'training sequences')

In [ ]:
BATCH, ACCUM = 2, 8                          # effective batch = 16 sequences = 8,192 tokens per optimizer step
steps_per_epoch = math.ceil(len(train_ds) / (BATCH * ACCUM))
MAX_STEPS = -1                               # -1 = one full epoch. Set e.g. 100 for a quick trial run.
total_steps = steps_per_epoch if MAX_STEPS < 0 else MAX_STEPS
print(f'{steps_per_epoch} steps per epoch; running {total_steps} steps')

model.config.use_cache = False               # the KV cache is for generation, not training
args = TrainingArguments(
    output_dir=str(WORKDIR / 'cpt_runs'),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    learning_rate=2e-5,                      # small: we nudge a trained model, we do not retrain it
    lr_scheduler_type='cosine',
    warmup_steps=max(1, int(0.05 * total_steps)),
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=1,                         # capture loss at every step for the curve
    save_strategy='no',
    bf16=BF16_OK,                            # mixed precision only where the GPU supports it
    gradient_checkpointing=True,
    report_to='none',
    seed=SEED,
)
recorder = LossRecorder()
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=default_data_collator, callbacks=[recorder])
t0 = time.time()
trainer.train()
print(f'training time: {(time.time() - t0) / 60:.1f} min')
assert len(recorder.losses) > 0, 'TODO 11: the callback recorded nothing'

In [ ]:
losses = np.array(recorder.losses)
smooth = pd.Series(losses).rolling(10, min_periods=1).mean()

plt.figure(figsize=(9, 4))
plt.plot(recorder.steps, losses, alpha=.35, label='loss per step')
plt.plot(recorder.steps, smooth, lw=2, label='10-step moving average')
plt.xlabel('optimizer step'); plt.ylabel('cross-entropy loss'); plt.title('CPT training loss')
plt.grid(alpha=.3); plt.legend(); plt.show()

print(f'first loss {losses[0]:.3f}  ->  last 10-step mean {losses[-10:].mean():.3f}')
# a rough plateau marker: first step where the moving average is within 5% of its final value
plateau = next(s for s, v in zip(recorder.steps, smooth) if v <= smooth.iloc[-1] * 1.05)
print('loss is within 5% of its final value from step', plateau)
if losses[0] > 8:
    print('⚠ starting loss is ~ln(vocab): the model looks randomly initialised. Re-check Step 3.')

In [ ]:
model.save_pretrained(CPT_DIR)
tok.save_pretrained(CPT_DIR)
print('saved CPT model to', CPT_DIR)

del trainer; gc.collect(); torch.cuda.empty_cache()

#### ✍️ Your inference (mandatory, marks depend on it)

- What was the starting loss? Why is ~2-4 expected for a pretrained model, while ~11-12 means random weights? (Hint: ln(vocab size).)
- Where does the curve plateau? What does a plateau mean?
- Justify the learning rate, warmup and number of steps. What would you change if the loss spiked?
- How long did training take, and on which GPU?

_Write your answer here._

---
## Step 5: Evaluation: perplexity and catastrophic forgetting [2 marks] · *explainer §5, §6*

**Perplexity (PPL) = e^(mean cross-entropy loss).** Read it as *"the model is as unsure as if it were choosing between
PPL equally likely next tokens"*. Lower is better. We compute it on the held-out eval blocks, with no gradients and no training.

In [ ]:
@torch.no_grad()
def perplexity(m, blocks, batch_size=4, max_blocks=300):
    m.eval()
    blocks = blocks[:max_blocks]
    losses = []
    for i in range(0, len(blocks), batch_size):
        x = torch.tensor(blocks[i:i + batch_size], device=m.device)
        losses.append(m(input_ids=x, labels=x).loss.item())
    # TODO 12: mean_loss = the average of `losses`;  ppl = e to the power mean_loss (math.exp)
    mean_loss = None
    ppl = None
    return ppl, mean_loss

eval_blocks = pd.read_parquet(DATA_DIR / 'cpt_eval.parquet')['input_ids'].tolist()
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)
cpt_model  = AutoModelForCausalLM.from_pretrained(CPT_DIR,  dtype=torch.float32).to(DEVICE)

base_ppl, base_loss = perplexity(base_model, eval_blocks)
cpt_ppl,  cpt_loss  = perplexity(cpt_model,  eval_blocks)
# TODO 13: percentage drop in perplexity from base to CPT
ppl_drop = None

assert base_ppl is not None and abs(base_ppl - math.exp(base_loss)) < 1e-6, 'TODO 12'
pd.DataFrame([['base model', base_loss, base_ppl], ['CPT model', cpt_loss, cpt_ppl]],
             columns=['model', 'eval loss', 'perplexity']).round(3)

In [ ]:
print(f'Perplexity reduction: {ppl_drop:.1f}%   (a typical successful CPT run: 10-40%)')

# CPT on the 3 domain prompts, next to the baseline from Step 3
baseline['cpt_model_output'] = [generate(cpt_model, p) for p in DOMAIN_PROMPTS]
baseline.to_csv(DATA_DIR / 'baseline_outputs.csv', index=False)
baseline

### 5B: Catastrophic forgetting check
Did reading only Linux manuals make the model forget general knowledge? Run 3 general prompts through both models,
read the outputs, and fill in the verdict yourself: **Retained** or **Degraded**.

In [ ]:
forget = pd.DataFrame({
    'prompt': GENERAL_PROMPTS,
    'base_model': [generate(base_model, p, max_new_tokens=30) for p in GENERAL_PROMPTS],
    'cpt_model':  [generate(cpt_model,  p, max_new_tokens=30) for p in GENERAL_PROMPTS],
})
# TODO 14: after reading the outputs, write one verdict per prompt: 'Retained' or 'Degraded'
forget['verdict'] = ['?', '?', '?']
assert set(forget['verdict']) <= {'Retained', 'Degraded'}, 'TODO 14: fill in the verdicts'
forget

In [ ]:
del base_model, cpt_model, model; gc.collect(); torch.cuda.empty_cache()

#### ✍️ Your inference (mandatory, marks depend on it)

- Report base PPL, CPT PPL and the % reduction. Is it in the expected 10-40% range? If not, why might it be (corpus size, steps, LR)?
- Compare the base and CPT outputs on the 3 domain prompts: what changed (terminology, commands, style)?
- Was general knowledge Retained or Degraded? Explain using the learning rate and number of steps.
- If it had degraded, name two fixes (CS-2: lower LR / fewer steps, replay of general data).
- Why does lower domain PPL *not* automatically mean the model answers questions well? (Leads into Part B.)

_Write your answer here._

---
# PART B: Instruction fine-tuning with QLoRA

## B1: Instruction dataset creation [2 marks] · *explainer §7, §8*

**Method: heuristic (template conversion, CS-3 slide 8).** The RHEL guides are organised as numbered sections such as
`1.7.3. Assigning a network interface to a zone`. Each section is a ready-made Q&A pair:
- **instruction** = a question built from the section title (*"Walk me through assigning a network interface to a zone on RHEL 9."*)
- **response** = the first ~160 words of that section's text: grounded, with no outside facts

This is free, repeatable and needs no API key. The cost is some clumsy questions and responses cut mid-thought. Discuss
that in your inference. (The *optional* LLM route and its exact prompt template are given at the end of this section.)

In [ ]:
SECTION = re.compile(r'^(\d+(?:\.\d+)+)\.?\s+([A-Z][^\n]{3,100})$')
SKIP_TITLES = ('additional resources', 'prerequisites', 'verification', 'next steps', 'procedure')

def sections(text):
    secs, cur = [], None
    for line in text.splitlines():
        line = line.strip()
        m = SECTION.match(line)
        if m:
            cur = [m.group(2), []]; secs.append(cur)
        elif cur and not cur[1] and cur[0].isupper() and line.isupper():
            cur[0] += ' ' + line            # an ALL-CAPS title that wrapped onto a second line
        elif cur:
            cur[1].append(line)
    return secs

def make_pair(title, body_words, product):
    title = title.capitalize() if title.isupper() else title
    t = title[0].lower() + title[1:]
    rng = random.Random(title)            # same title -> same template, so the dataset is reproducible
    if title.split()[0].lower().endswith('ing'):
        q = rng.choice([f'Walk me through {t} on {product}.', f'What are the steps for {t} on {product}?',
                        f'How do I go about {t} on {product}?'])
    else:
        q = rng.choice([f'Explain {t} in {product}.', f'What should an admin know about {t} on {product}?'])
    resp = ' '.join(body_words[:160])
    cut = resp.rfind('. ')
    if cut > 200:
        resp = resp[:cut + 1]             # end on a full sentence
    return q, resp

pairs = []
for f in sorted(CORPUS_DIR.glob('*.txt')):
    product = 'Ubuntu Server' if 'ubuntu' in f.stem else 'RHEL 9'
    for title, body in sections(f.read_text(encoding='utf-8')):
        words = ' '.join(body).split()
        if len(words) < 40 or title.lower().startswith(SKIP_TITLES):
            continue
        q, a = make_pair(title, words, product)
        pairs.append({'instruction': q, 'response': a, 'source': f.stem})

# drop repeated questions, keep the first one
seen_q = set()
pairs = [p for p in pairs if not (p['instruction'] in seen_q or seen_q.add(p['instruction']))]
print(len(pairs), 'instruction-response pairs')

In [ ]:
MAX_PAIRS = 2000      # justify: 0.5B model, T4 time budget, ~250 tokens per pair
random.Random(SEED).shuffle(pairs)
pairs = pairs[:MAX_PAIRS]

# TODO 15: write `pairs` to DATA_DIR / 'instruction_dataset.jsonl' with one JSON object per line,
#          keeping only the keys 'instruction' and 'response'.
#          Hint: f.write(json.dumps({...}, ensure_ascii=False) + '\n')
with open(DATA_DIR / 'instruction_dataset.jsonl', 'w', encoding='utf-8') as f:
    pass

# TODO 16: split 80/20. `pairs` is already shuffled with SEED, so slice it.
n_train = None
train_pairs, eval_pairs = None, None

rows = [json.loads(l) for l in open(DATA_DIR / 'instruction_dataset.jsonl', encoding='utf-8')]
assert len(rows) == len(pairs) and set(rows[0]) == {'instruction', 'response'}, 'TODO 15'
assert len(train_pairs) + len(eval_pairs) == len(pairs) and abs(len(train_pairs) / len(pairs) - 0.8) < 0.01, 'TODO 16'

tok_lens = [len(tok(p['instruction'] + ' ' + p['response'])['input_ids']) for p in pairs]
print(f'train: {len(train_pairs)}   eval: {len(eval_pairs)}   avg tokens/pair: {np.mean(tok_lens):.0f}   max: {max(tok_lens)}')
pd.DataFrame(pairs[:5])[['instruction', 'response']]

<details><summary><b>Optional: synthetic generation with an external LLM (the exact prompt template)</b></summary>

If you use an LLM instead, the assignment requires you to include the exact prompt. Use this one per ~800-word chunk of a
`domain_corpus/*.txt` file:

```
Read the text below and generate 10 instruction-response pairs in JSON format based ONLY on this text.
Each entry must have "instruction" and "response" keys. Instructions should read like real helpdesk
questions from a Linux administrator. Responses must use only facts, commands and file paths that
appear in the text. Return a JSON list and nothing else.

TEXT:
<chunk>
```
Then parse the JSON, drop entries whose response is not supported by the chunk, and continue from TODO 15.
</details>

#### ✍️ Your inference (mandatory, marks depend on it)

- Which method did you use and why? What are its weaknesses (look at the 5 samples)?
- Justify the dataset size: number of pairs, average tokens per pair, and the T4 time budget.
- Report the train / eval counts and the seed.
- How did you keep responses grounded (no outside facts)?

_Write your answer here._

## B2: QLoRA fine-tuning with one adapter [2 marks] · *explainer §9, §10*

**QLoRA** = the CPT model stored frozen in **4-bit NF4** + small trainable **LoRA** matrices (CS-3 slides 21-22).
On a T4 the compute dtype is fp16, because the T4 has no bf16 (CS-3 slide 22 says the same).

We train **Adapter B (Balanced)**: `r = 16`, `lora_alpha = 32`, `target_modules = q_proj, v_proj`.
You may pick A or C instead; just say why in your inference.

`SFTTrainer` reads the `messages` column and formats each example with the model's **chat template** (explainer §8).

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)

sft_tok = AutoTokenizer.from_pretrained(CPT_DIR)
qmodel = AutoModelForCausalLM.from_pretrained(CPT_DIR, quantization_config=bnb_cfg, device_map={'': 0})
qmodel = prepare_model_for_kbit_training(qmodel, use_gradient_checkpointing=True)
print(f'4-bit model memory: {qmodel.get_memory_footprint() / 1e9:.2f} GB')

# TODO 17: fill in Adapter B from the assignment table: r=16, lora_alpha=32, target_modules=['q_proj', 'v_proj']
lora_cfg = LoraConfig(
    r=None,
    lora_alpha=None,
    target_modules=None,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
qmodel = get_peft_model(qmodel, lora_cfg)
qmodel.print_trainable_parameters()

In [ ]:
to_chat = lambda p: {'messages': [{'role': 'user', 'content': p['instruction']},
                                  {'role': 'assistant', 'content': p['response']}]}
train_sft = Dataset.from_list([to_chat(p) for p in train_pairs])
eval_sft  = Dataset.from_list([to_chat(p) for p in eval_pairs])
print(sft_tok.apply_chat_template(train_sft[0]['messages'], tokenize=False))   # what the model actually sees

sft_args = SFTConfig(
    output_dir=str(WORKDIR / 'sft_runs'),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=2e-4,                  # LoRA matrices start at zero, so they need a much larger LR than CPT
    lr_scheduler_type='cosine',
    warmup_steps=10,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='no',
    max_length=512,
    bf16=BF16_OK, fp16=not BF16_OK,
    gradient_checkpointing=True,
    report_to='none',
    seed=SEED,
)
sft_trainer = SFTTrainer(model=qmodel, args=sft_args, train_dataset=train_sft,
                         eval_dataset=eval_sft, processing_class=sft_tok)
t0 = time.time()
sft_trainer.train()
print(f'QLoRA training time: {(time.time() - t0) / 60:.1f} min,  peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

hist = pd.DataFrame(sft_trainer.state.log_history)
display(hist[['epoch', 'loss', 'eval_loss']].dropna(how='all', subset=['loss', 'eval_loss']).tail(10))
qmodel.save_pretrained(ADAPTER_DIR)
print('adapter saved to', ADAPTER_DIR, '- size:',
      round(sum(f.stat().st_size for f in ADAPTER_DIR.glob('*')) / 1e6, 1), 'MB')

#### ✍️ Your inference (mandatory, marks depend on it)

- Which adapter did you choose and why? Report the trainable parameter count and % from print_trainable_parameters().
- Report the key hyperparameters (batch, LR, epochs, max length) and justify the LR being 10x higher than CPT.
- Compare train loss vs eval loss: overfitting or not?
- How big is the adapter file compared with the full model? Why does that matter in production (CS-3 slide 19)?

_Write your answer here._

## B3: Evaluation analysis [1 mark]
The same 3 domain prompts, now wrapped in the chat template, compared with the base and CPT outputs.

In [ ]:
baseline['cpt_plus_adapter_B'] = [generate(qmodel, p, chat=True) for p in DOMAIN_PROMPTS]
baseline.to_csv(DATA_DIR / 'final_comparison.csv', index=False)
baseline

#### ✍️ Your inference (mandatory, marks depend on it)

- For each prompt: is it more correct, more complete, using better domain terms? Any hallucinated commands or options?
- Does the adapter model *answer* the question where the base model just continued the text? Why (instruction tuning, CS-3 slide 7)?
- One concrete way to improve the results (more or better pairs, Adapter C, more epochs, an LLM-generated dataset...).

_Write your answer here._

---
## Submission checklist
1. This notebook **with outputs**: *File → Download → .ipynb*.
2. HTML export: run `jupyter nbconvert --to html practice.ipynb` on your laptop (or in Colab: upload the .ipynb and run the same command).
3. `instruction_dataset.jsonl`: in `WORKDIR/data/`.
4. `domain_corpus/*.txt`: zip it with the cell below.

Before you export, check that every ✍️ inference cell has your own text.

In [ ]:
import shutil
shutil.make_archive(str(WORKDIR / 'domain_corpus'), 'zip', CORPUS_DIR)
print('ready to download:', WORKDIR / 'domain_corpus.zip', 'and', DATA_DIR / 'instruction_dataset.jsonl')